# 강의 03 · 실습 0 — 에이전트 동작 원리 · 손 루프 · (2-1) 빈칸 채우기 I

## 1. 문제상황

- 구름월드 놀이공원 고객센터에는 운영시간, 주차 요금, 환불 규정을 묻는 질문이 하루 종일 들어옵니다.
- 질문 중에는 「지금 몇 시예요」처럼 FAQ 문서가 아니라 현재 시각을 봐야 답할 수 있는 질문도 섞여 있습니다.
- 담당자는 질문마다 FAQ 문서를 찾아보거나 시계를 본 뒤 답을 씁니다.
- 언어 모델은 구름월드의 FAQ 내용도 현재 시각도 알지 못하므로, 모델에게 질문을 그대로 넘기면 모르는 내용을 지어내거나 답하지 못합니다.

## 2. 문제와 목표

- **문제**: 모델은 구름월드의 FAQ 내용과 현재 시각을 모릅니다. 사람이 질문마다 FAQ 조회와 시각 확인을 대신 해 주어야 합니다.
- **목표**
  - 질문을 입력하면 모델이 두 도구 중 필요한 도구를 스스로 골라 호출 요청을 보내야 합니다.
    - 두 도구: FAQ 조회(`faq_lookup`), 현재 시각(`get_now`)
  - 우리 코드가 도구를 실행해 결과를 되먹이고, 모델이 최종 답을 만드는 안내 프로그램을 프레임워크 없이 만듭니다.
- **목표 달성 여부의 판정 기준**:
  - 운영시간 질문에서는 FAQ 조회 도구가, 현재 시각 질문에서는 현재 시각 도구가 호출되고,
  - 두 질문 모두 도구 결과가 반영된 최종 답으로 끝나는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

1. **도구를 선언합니다.**
    - FAQ 사전(운영시간·주차·환불)에서 항목을 조회하는 `faq_lookup(topic)`과 현재 날짜·시각을 돌려주는 `get_now()` 두 함수를 만들고, 도구 이름으로 함수를 찾는 표 `TOOLS`를 만듭니다.
    - FAQ 사전의 값은 「6. 코드 — 스텝바이스텝」 단계 ①의 코드에 있습니다.
2. **도구 스키마를 씁니다.**
    - 두 도구의 이름(`name`)·설명(`description`)·파라미터(`parameters`, JSON Schema)를 담은 목록 `TOOL_SCHEMA`를 씁니다.
    - `faq_lookup`의 설명은 「시설 FAQ에서 항목을 조회한다. 운영시간, 주차, 환불 질문에 쓴다.」, `topic` 설명은 「조회할 항목 이름. 운영시간, 주차, 환불 중 하나.」입니다.
3. **모델을 호출하고 도구 호출 요청을 판정합니다.**
    - 사용자 질문을 `{"role": "user", "content": 질문}`으로 담은 대화 기록을 `completion(model=MODEL, messages=messages, tools=TOOL_SCHEMA)`로 보내고, `res.choices[0].message.tool_calls`가 비어 있는지로 판정합니다.
4. **도구 결과를 되먹여 반복합니다.**
    - 도구 호출 요청이 있으면 모델 응답 메시지를 대화 기록에 붙이고, 요청마다 `TOOLS`에서 함수를 찾아 `json.loads(call.function.arguments)`로 푼 인자로 실행한 뒤, 결과를 `{"role": "tool", "tool_call_id": call.id, "content": 결과}`로 붙이고 모델을 다시 호출합니다.
    - 반복은 4회를 상한으로 하고, 상한에 닿으면 「반복 한도 초과」를 돌려줍니다.
5. **두 질문으로 실행합니다.**
    - 「운영시간이 어떻게 되나요?」와 「지금 몇 시인가요?」를 차례로 넣어, 질문마다 호출된 도구 이름과 인자, 도구 결과, 최종 답을 화면에 출력합니다.
    - 출력 줄의 이름은 `[도구 호출]`·`[도구 결과]`·`[최종 답]`입니다.

## 5. 코드 골격 — 도구 호출 루프 4단(손 루프)

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 대화 기록 초기화 | 사용자 질문을 첫 메시지로 넣습니다 | `messages = [user_msg(question)]` | 3 |
| ② 도구 목록과 함께 모델 호출 | 스키마를 붙여 모델을 부릅니다 | `completion(model=, messages=, tools=TOOL_SCHEMA)` | 2, 3 |
| ③ 도구 호출 여부 판정 | 응답에 호출 요청이 실렸는지 봅니다 | `msg.tool_calls` | 3 |
| ④ 도구 실행 결과 되먹임 | 함수를 실행하고 결과를 기록에 붙여 다시 호출합니다 | `TOOLS[name](**json.loads(args))`, `tool_msg(call.id, result)` | 1, 4, 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델 이름을 정합니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.

In [ ]:
import ____
import os
from datetime import datetime

from dotenv import load_dotenv, find_dotenv
from litellm import ____

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

MODEL = "openai/gpt-5.6-luna"
print("준비를 마쳤습니다.")

### 단계 ① — 도구 두 개 등록 (요구사항 1)

- 도구의 실체는 평범한 파이썬 함수입니다. `TOOLS`는 도구 이름과 실제 함수를 짝지은 사전이며, 모델이 보낸 이름을 여기서 함수로 바꿉니다.

In [ ]:
FAQ = {
    "운영시간": "매일 09:30~21:00에 운영합니다.",
    "주차": "주차장은 4,000대 규모이며 최초 30분은 무료입니다.",
    "환불": "이용일 전날까지 전액 환불, 당일은 50% 환불입니다.",
}


# 입력: topic(조회할 항목 이름. 운영시간, 주차, 환불 중 하나)
# 기능: FAQ 사전에서 topic 항목을 찾는다.
# 반환: 그 항목의 안내 문장
def faq_lookup(topic: str) -> str:
    """FAQ에서 항목을 조회한다."""
    return ____


# 입력: 없음
# 기능: 현재 시각을 읽어 "YYYY-MM-DD HH:MM" 형식의 문자열로 바꾼다.
# 반환: 현재 날짜와 시각 문자열
def get_now() -> str:
    """현재 날짜와 시각을 반환한다."""
    return ____


# 도구 이름 → 실제 함수. 모델이 보낸 이름을 여기서 함수로 바꿉니다.
TOOLS = {____}
print("등록한 도구:", list(TOOLS))

### 단계 ② — 도구 스키마 (요구사항 2)

- 스키마는 함수를 모델에게 소개하는 명세입니다. 이름은 모델이 호출 요청에 적어 보내는 식별자, 설명은 모델이 도구를 고를 때 읽는 유일한 근거, 파라미터는 인자의 이름·타입·필수 여부입니다. **설명의 질이 도구 선택의 질을 결정합니다.**

In [ ]:
TOOL_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": ____,   # 모델이 호출 요청에 적어 보내는 식별자. TOOLS의 키와 같아야 합니다.
            "description": ____,   # 무엇을 하는 도구이고 언제 쓰는지. 모델이 도구를 고르는 유일한 근거입니다.
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": ____,   # 인자가 받는 값의 형식과 범위
                    }
                },
                "required": [____],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": ____,
            "description": ____,
            "parameters": {"type": "object", "properties": {}},
        },
    },
]


# 입력: q(사용자 질문 문자열) / 기능: 사용자 메시지 딕셔너리를 만든다 / 반환: {"role": ..., "content": ...}
def user_msg(q):
    return {"role": ____, "content": q}


# 입력: call_id(호출 id), content(도구 결과) / 기능: 도구 결과 메시지를 만든다 / 반환: role이 tool인 딕셔너리
def tool_msg(call_id, content):
    return {"role": ____, "tool_call_id": ____, "content": str(content)}


print("스키마에 실린 도구:", [t["function"]["name"] for t in TOOL_SCHEMA])

### 단계 ③ — 첫 호출과 판정 (요구사항 3)

- 아래 셀은 루프를 돌리기 전에 첫 호출의 응답을 그대로 열어 봅니다. 인자는 JSON 문자열로 돌아오므로 `json.loads`로 풉니다.

In [ ]:
messages = [user_msg("운영시간이 어떻게 되나요?")]
res = completion(model=____, messages=____, tools=____)
msg = res.choices[0].message
print("tool_calls:", msg.tool_calls)
# 여기에 msg.tool_calls가 비어 있는지로 판정해, 있으면 도구 이름과 json.loads로 푼 인자를, 없으면 「최종 답」임을 출력하는 분기를 작성합니다.

### 단계 ④ — 도구 호출 루프 (요구사항 4, 5)

- 에이전트의 실체는 반복문 하나입니다. 호출 → 판정 → 실행·되먹임을 `for` 문 안에 넣고, 종료 조건은 도구 호출 요청이 없는 응답과 반복 한도 도달 둘입니다.

In [ ]:
def run_agent(question, max_turn=4):
    messages = [user_msg(question)]
    for _ in range(max_turn):
        res = completion(model=MODEL, messages=messages, tools=TOOL_SCHEMA)
        msg = res.choices[0].message
        if not ____:            # 판정: 도구 호출 요청이 없으면 최종 답
            return ____
        messages.append(____)   # 모델 응답을 대화 기록에 붙인다
        for call in msg.tool_calls:
            fn = TOOLS[____]
            args = ____         # JSON 문자열 인자를 딕셔너리로
            print(f"  [도구 호출] {call.function.name}({args})")
            result = fn(**args)
            print(f"  [도구 결과] {result}")
            messages.append(____)   # 도구 결과를 role이 tool인 메시지로 붙인다
    return "반복 한도 초과"


# 여기에 질문 두 개(「운영시간이 어떻게 되나요?」「지금 몇 시인가요?」)를 차례로 run_agent에 넣어
# 「=== N번 질문: … ===」 줄과 「[최종 답]」 줄을 출력하는 반복문을 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ③의 출력에서 첫 호출의 응답에 도구 호출 요청이 있고, 도구 이름이 `faq_lookup`, 인자가 `{'topic': '운영시간'}`입니다.
2. 1번 질문(운영시간)에서는 `[도구 호출] faq_lookup({'topic': '운영시간'})`과 `[도구 결과] 매일 09:30~21:00에 운영합니다.`가 찍힌 뒤 최종 답이 나오고, 최종 답에 그 운영시간이 들어 있습니다.
3. 2번 질문(현재 시각)에서는 `[도구 호출] get_now({})`가 찍히고, 최종 답에 도구 결과의 날짜와 시각이 들어 있습니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 `lec03_ex00_s1.ipynb`와 대조해 채운 빈칸을 고칩니다.